In [ ]:
# Use the current Dash class with built-in Jupyter support
from dash import Dash, Input, Output, ctx, dash_table, dcc, html, no_update

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
import plotly.express as px
import base64
import logging
import math
import os
import re
from pathlib import Path

# Pandas organizes the animal records used by the dashboard
import pandas as pd
from pymongo.errors import PyMongoError

from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# Log useful events and errors without changing the dashboard behavior
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger('grazioso_dashboard')

# Connect through the same CRUD class used by the original project
database_error = ''
total_record_count = 0
breed_names = []
maximum_age_weeks = 156
try:
    username = os.getenv('AAC_DB_USERNAME')
    password = os.getenv('AAC_DB_PASSWORD')
    if not username or not password:
        raise ValueError('Database credentials are not configured.')

    db = AnimalShelter(username, password)
    db.client.admin.command('ping')
    db.ensure_rescue_index()
    total_record_count = db.count({})
    df = pd.DataFrame.from_records(db.read({}, limit=10))
    df.drop(columns=['_id'], inplace=True, errors='ignore')
    breed_names = sorted(
        str(breed) for breed in db.distinct('breed') if breed
    )
    oldest_records = db.read(
        {},
        projection={'age_upon_outcome_in_weeks': True, '_id': False},
        sort=[('age_upon_outcome_in_weeks', -1)],
        limit=1
    )
    if oldest_records:
        maximum_age_weeks = max(
            0,
            math.ceil(float(oldest_records[0]['age_upon_outcome_in_weeks']))
        )
    logger.info('Connected to %s animal records.', total_record_count)
except ValueError:
    logger.exception('Database configuration is unavailable.')
    db = None
    df = pd.DataFrame()
    database_error = 'Database credentials are not configured.'
except (PyMongoError, TypeError, KeyError, OverflowError):
    logger.exception('The dashboard could not connect to MongoDB.')
    db = None
    df = pd.DataFrame()
    database_error = 'The database could not be reached. Check MongoDB and try again.'

# Keep rescue rules in one dictionary so they are easy to validate and reuse
RESCUE_PROFILES = {
    'water': {
        'animal_type': 'Dog',
        'breeds': [
            'Labrador Retriever Mix',
            'Chesapeake Bay Retriever',
            'Newfoundland'
        ],
        'min_age': 26,
        'max_age': 156,
        'ideal_min_age': 52,
        'ideal_max_age': 104
    },
    'mountain': {
        'animal_type': 'Dog',
        'breeds': [
            'German Shepherd',
            'Alaskan Malamute',
            'Siberian Husky',
            'Rottweiler',
            'Border Collie'
        ],
        'min_age': 26,
        'max_age': 156,
        'ideal_min_age': 52,
        'ideal_max_age': 104
    },
    'disaster': {
        'animal_type': 'Dog',
        'breeds': [
            'Doberman Pinscher',
            'German Shepherd',
            'Golden Retriever',
            'Bloodhound',
            'Rottweiler'
        ],
        'min_age': 26,
        'max_age': 156,
        'ideal_min_age': 52,
        'ideal_max_age': 104
    }
}

FILTER_NAMES = {
    'reset': 'all available shelter animals',
    'water': 'the Water Rescue profile',
    'mountain': 'the Mountain or Wilderness Rescue profile',
    'disaster': 'the Disaster or Individual Tracking profile'
}


def build_rescue_query(filter_type):
    """Validate a rescue profile and build its MongoDB query."""
    if filter_type == 'reset':
        return {}

    profile = RESCUE_PROFILES.get(filter_type)
    if profile is None:
        logger.warning("Unknown rescue filter '%s'; Reset will be used.", filter_type)
        return {}

    required_fields = {
        'animal_type',
        'breeds',
        'min_age',
        'max_age',
        'ideal_min_age',
        'ideal_max_age'
    }
    if not required_fields.issubset(profile):
        raise ValueError(f"The {filter_type} rescue profile is incomplete.")
    if not profile['breeds']:
        raise ValueError(f"The {filter_type} rescue profile has no breeds.")
    if profile['min_age'] > profile['max_age']:
        raise ValueError(f"The {filter_type} rescue age range is invalid.")

    return {
        'animal_type': profile['animal_type'],
        'breed': {'$in': profile['breeds']},
        'age_upon_outcome_in_weeks': {
            '$gte': profile['min_age'],
            '$lte': profile['max_age']
        }
    }


def build_database_query(
    filter_type,
    selected_breed=None,
    age_range=None,
    require_location=False,
    table_filter_query=''
):
    """Combine rescue and optional filters into one MongoDB query."""
    conditions = []
    rescue_query = build_rescue_query(filter_type)
    if rescue_query:
        conditions.append(rescue_query)
    if selected_breed:
        conditions.append({'breed': selected_breed})
    if age_range is not None:
        conditions.append({
            'age_upon_outcome_in_weeks': {
                '$gte': age_range[0],
                '$lte': age_range[1]
            }
        })
    if require_location:
        conditions.extend([
            {'location_lat': {'$type': 'number', '$gte': -90, '$lte': 90}},
            {'location_long': {'$type': 'number', '$gte': -180, '$lte': 180}}
        ])
    conditions.extend(parse_table_filter_query(table_filter_query))
    if not conditions:
        return {}
    if len(conditions) == 1:
        return conditions[0]
    return {'$and': conditions}


DATABASE_SORT_FIELDS = {
    'breed': 'breed',
    'age': 'age_upon_outcome_in_weeks',
    'name': 'name'
}


def database_sort(sort_field):
    """Return a MongoDB sort for fields stored in the collection."""
    field = DATABASE_SORT_FIELDS.get(sort_field)
    return [(field, 1)] if field else None


def coordinates_are_valid(latitude, longitude):
    """Return True when latitude and longitude are within valid ranges."""
    return -90 <= latitude <= 90 and -180 <= longitude <= 180


def record_has_valid_location(record):
    """Return True when a record contains usable map coordinates."""
    try:
        latitude = float(record.get('location_lat'))
        longitude = float(record.get('location_long'))
    except (AttributeError, TypeError, ValueError):
        return False
    return coordinates_are_valid(latitude, longitude)


def normalize_age_range(age_range):
    """Return a valid low-to-high age range or None."""
    if not age_range or len(age_range) != 2:
        return None
    try:
        low_age = float(age_range[0])
        high_age = float(age_range[1])
    except (TypeError, ValueError):
        return None
    return (low_age, high_age) if low_age <= high_age else (high_age, low_age)


def candidate_matches_filters(
    record,
    profile=None,
    selected_breed=None,
    age_range=None,
    require_location=False
):
    """Validate one animal against the selected profile and optional filters."""
    if not isinstance(record, dict):
        return False

    breed = record.get('breed')
    animal_type = record.get('animal_type')
    normalized_age = normalize_age_range(age_range)

    try:
        age = float(record.get('age_upon_outcome_in_weeks'))
    except (TypeError, ValueError):
        age = None

    if profile is not None:
        if animal_type != profile['animal_type']:
            return False
        if breed not in profile['breeds']:
            return False
        if age is None or not profile['min_age'] <= age <= profile['max_age']:
            return False

    if selected_breed and breed != selected_breed:
        return False

    if normalized_age is not None:
        low_age, high_age = normalized_age
        if age is None or not low_age <= age <= high_age:
            return False

    if require_location and not record_has_valid_location(record):
        return False

    return True


def calculate_suitability_score(record, profile):
    """Score a rescue candidate using breed, age, and location information."""
    if profile is None:
        return None

    score = 0
    if record.get('breed') in profile['breeds']:
        score += 40

    try:
        age = float(record.get('age_upon_outcome_in_weeks'))
    except (TypeError, ValueError):
        age = None

    if age is not None:
        if profile['ideal_min_age'] <= age <= profile['ideal_max_age']:
            score += 40
        elif profile['min_age'] <= age <= profile['max_age']:
            score += 30

    if record_has_valid_location(record):
        score += 20

    return score


def age_sort_value(record):
    """Return a numeric age for sorting and place invalid ages last."""
    try:
        return float(record.get('age_upon_outcome_in_weeks'))
    except (AttributeError, TypeError, ValueError):
        return float('inf')


def sort_candidates(candidates, sort_field, has_rescue_profile):
    """Return a new list ordered by the selected field."""
    if sort_field == 'breed':
        return sorted(
            candidates,
            key=lambda item: (
                not bool(item.get('breed')),
                str(item.get('breed') or '').lower()
            )
        )
    if sort_field == 'age':
        return sorted(
            candidates,
            key=age_sort_value
        )
    if sort_field == 'name':
        return sorted(
            candidates,
            key=lambda item: (
                not bool(item.get('name')),
                str(item.get('name') or '').lower()
            )
        )
    if sort_field == 'suitability' and has_rescue_profile:
        return sorted(
            candidates,
            key=lambda item: item.get('suitability_score') or 0,
            reverse=True
        )
    return candidates


def process_candidates(
    records,
    filter_type,
    selected_breed=None,
    age_range=None,
    require_location=False,
    sort_field='suitability'
):
    """Filter, score, and sort animal records for the dashboard."""
    profile = RESCUE_PROFILES.get(filter_type)
    candidates = []

    for record in records:
        if not candidate_matches_filters(
            record,
            profile,
            selected_breed,
            age_range,
            require_location
        ):
            continue

        candidate = dict(record)
        candidate['suitability_score'] = calculate_suitability_score(
            candidate,
            profile
        )
        candidates.append(candidate)

    return sort_candidates(candidates, sort_field, profile is not None)


# Add the score column without removing any original data columns
if 'suitability_score' not in df.columns:
    df['suitability_score'] = None

DATABASE_FIELDS = set(df.columns) - {'suitability_score'}


def parse_filter_value(value_text):
    """Convert one table-filter value to text or a number."""
    value_text = value_text.strip()
    if len(value_text) >= 2 and value_text[0] == value_text[-1] \
            and value_text[0] in ('"', "'", '`'):
        return value_text[1:-1].replace('\\' + value_text[0], value_text[0])
    try:
        number = float(value_text)
        return int(number) if number.is_integer() else number
    except ValueError:
        return value_text


def parse_table_filter_query(filter_query):
    """Convert Dash table filters into validated MongoDB conditions."""
    if not filter_query:
        return []

    operators = [
        ('>=', '$gte'),
        ('<=', '$lte'),
        ('!=', '$ne'),
        ('>', '$gt'),
        ('<', '$lt'),
        ('=', '$eq'),
        ('ge', '$gte'),
        ('le', '$lte'),
        ('ne', '$ne'),
        ('gt', '$gt'),
        ('lt', '$lt'),
        ('eq', '$eq'),
        ('scontains', 'scontains'),
        ('sdatestartswith', 'sdatestartswith'),
        ('contains', 'contains'),
        ('datestartswith', 'datestartswith')
    ]
    conditions = []
    for filter_part in filter_query.split(' && '):
        parsed = None
        for token, mongo_operator in operators:
            marker = f' {token} '
            if marker not in filter_part:
                continue
            name_part, value_part = filter_part.split(marker, 1)
            column = name_part.strip().strip('{}')
            if column not in DATABASE_FIELDS:
                raise ValueError(f'Unsupported table filter: {column}')
            value = parse_filter_value(value_part)
            if mongo_operator == '$eq':
                parsed = {column: value}
            elif mongo_operator == 'contains':
                parsed = {column: {'$regex': re.escape(str(value)), '$options': 'i'}}
            elif mongo_operator == 'scontains':
                parsed = {column: {'$regex': re.escape(str(value))}}
            elif mongo_operator == 'datestartswith':
                parsed = {column: {'$regex': '^' + re.escape(str(value))}}
            elif mongo_operator == 'sdatestartswith':
                parsed = {column: {'$regex': '^' + re.escape(str(value))}}
            else:
                parsed = {column: {mongo_operator: value}}
            break
        if parsed is None:
            raise ValueError('A table filter could not be understood.')
        conditions.append(parsed)
    return conditions


def table_sort_settings(sort_field, sort_by):
    """Choose database or suitability sorting for the current table."""
    if sort_by:
        column = sort_by[0].get('column_id')
        direction = sort_by[0].get('direction', 'asc')
        if column == 'suitability_score':
            return None, 'suitability', direction
        if column not in DATABASE_FIELDS:
            raise ValueError(f'Unsupported table sort: {column}')
        mongo_direction = -1 if direction == 'desc' else 1
        return [(column, mongo_direction)], 'database', direction
    return database_sort(sort_field), sort_field, 'desc'


# Prepare filter options without loading the entire collection
BREED_OPTIONS = [{'label': breed, 'value': breed} for breed in breed_names]

# Animal ages cannot be negative, even if the source data contains an error
MIN_AGE_WEEKS = 0
MAX_AGE_WEEKS = maximum_age_weeks

# The database stores ages in weeks, so other units are converted to weeks
AGE_UNIT_WEEKS = {
    'weeks': 1,
    'months': 52 / 12,
    'years': 52
}


def build_age_options(age_unit):
    """Build age choices in the unit selected by the user."""
    unit = age_unit if age_unit in AGE_UNIT_WEEKS else 'weeks'
    maximum = math.ceil(MAX_AGE_WEEKS / AGE_UNIT_WEEKS[unit])
    return [
        {'label': f'{age} {unit}', 'value': age}
        for age in range(0, maximum + 1)
    ]


def convert_age_range_to_weeks(age_range, age_unit):
    """Convert a selected age range into the weeks stored by the database."""
    normalized_range = normalize_age_range(age_range)
    if normalized_range is None:
        return None
    unit = age_unit if age_unit in AGE_UNIT_WEEKS else 'weeks'
    weeks_per_unit = AGE_UNIT_WEEKS[unit]
    return tuple(age * weeks_per_unit for age in normalized_range)


def selected_age_ranges(age_unit, youngest_age, oldest_age):
    """Return the displayed ages and the database range in weeks."""
    display_range = normalize_age_range([youngest_age, oldest_age])
    unit = age_unit if age_unit in AGE_UNIT_WEEKS else 'weeks'
    maximum_for_unit = math.ceil(MAX_AGE_WEEKS / AGE_UNIT_WEEKS[unit])
    if display_range == (0, maximum_for_unit):
        return display_range, None, unit
    return display_range, convert_age_range_to_weeks(display_range, unit), unit


AGE_OPTIONS = build_age_options('weeks')
PAGE_SIZE = 10
MAX_SCORED_RESULTS = 5000


def run_self_checks():
    """Run small automated checks before the dashboard starts."""
    water_query = build_rescue_query('water')
    assert water_query['animal_type'] == 'Dog'
    assert build_rescue_query('unexpected') == {}
    assert coordinates_are_valid(30.75, -97.48)
    assert not coordinates_are_valid(200, -97.48)

    sample = {
        'animal_type': 'Dog',
        'breed': 'Labrador Retriever Mix',
        'age_upon_outcome_in_weeks': 78,
        'location_lat': 30.75,
        'location_long': -97.48,
        'name': 'Sample Dog'
    }
    water_profile = RESCUE_PROFILES['water']
    assert candidate_matches_filters(sample, water_profile)
    assert calculate_suitability_score(sample, water_profile) == 100
    assert MIN_AGE_WEEKS == 0
    assert convert_age_range_to_weeks([1, 2], 'years') == (52, 104)
    combined_query = build_database_query(
        'water',
        selected_breed='Labrador Retriever Mix',
        age_range=(52, 104)
    )
    assert '$and' in combined_query
    assert database_sort('breed') == [('breed', 1)]
    assert parse_table_filter_query('{animal_type} scontains Dog') == [
        {'animal_type': {'$regex': 'Dog'}}
    ]

    older_sample = dict(sample)
    older_sample['age_upon_outcome_in_weeks'] = 130
    ranked = process_candidates(
        [older_sample, sample],
        'water',
        sort_field='suitability'
    )
    assert ranked[0]['suitability_score'] == 100
    assert process_candidates([sample], 'water', selected_breed='Bloodhound') == []
    logger.info('Algorithm and database self-checks passed.')


run_self_checks()


#########################
# Dashboard Layout / View
#########################
app = Dash(__name__)

# Load the packaged Grazioso Salvare logo
image_path = Path('Grazioso Salvare Logo.png')
if image_path.is_file():
    with image_path.open('rb') as image_file:
        encoded_image = base64.b64encode(image_file.read()).decode('utf-8')
    logo_component = html.Img(
        src=f'data:image/png;base64,{encoded_image}',
        alt='Grazioso Salvare dog logo',
        style={'height': '90px', 'maxWidth': '100%'}
    )
else:
    logo_component = html.Div('Grazioso Salvare logo is unavailable.')

# Prepare the first status message shown to the user
if database_error:
    initial_message = database_error
    initial_color = '#9b1c1c'
else:
    initial_message = f'Displaying all {total_record_count:,} available shelter animals.'
    initial_color = '#185f37'
initial_page_count = (
    math.ceil(total_record_count / PAGE_SIZE) if total_record_count else 0
)

app.layout = html.Div([
    html.Center(html.B(html.H1('CS-340 Dashboard'))),

    html.Center(logo_component),
    html.Center(html.H4("Zachary Lecroy")),  # unique identifier

    html.Hr(),

    html.Div([
        html.H3("Filter Rescue Type"),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Reset (All)', 'value': 'reset'},
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'}
            ],
            value='reset',
            # Keep only the visible option text clickable
            labelStyle={
                'display': 'table',
                'padding': '4px 0',
                'cursor': 'pointer'
            }
        )
    ]),

    html.Div([
        html.Div([
            html.Label('Breed'),
            dcc.Dropdown(
                id='breed-filter',
                options=BREED_OPTIONS,
                value=None,
                placeholder='All breeds',
                clearable=True
            )
        ], style={'flex': '1 1 280px'}),

        html.Div([
            html.Label('Age Unit'),
            dcc.Dropdown(
                id='age-unit',
                options=[
                    {'label': 'Weeks', 'value': 'weeks'},
                    {'label': 'Months', 'value': 'months'},
                    {'label': 'Years', 'value': 'years'}
                ],
                value='weeks',
                clearable=False
            )
        ], style={'flex': '1 1 160px'}),

        html.Div([
            html.Label('Youngest Age'),
            dcc.Dropdown(
                id='youngest-age',
                options=AGE_OPTIONS,
                value=MIN_AGE_WEEKS,
                clearable=False
            )
        ], style={'flex': '1 1 200px'}),

        html.Div([
            html.Label('Oldest Age'),
            dcc.Dropdown(
                id='oldest-age',
                options=AGE_OPTIONS,
                value=MAX_AGE_WEEKS,
                clearable=False
            )
        ], style={'flex': '1 1 200px'}),

        html.Div([
            html.Label('Location'),
            dcc.Checklist(
                id='location-filter',
                options=[{
                    'label': ' Only animals with valid map coordinates',
                    'value': 'valid'
                }],
                value=[]
            )
        ], style={'flex': '1 1 280px'}),

        html.Div([
            html.Label('Sort Results'),
            dcc.Dropdown(
                id='sort-field',
                options=[
                    {'label': 'Rescue Suitability', 'value': 'suitability'},
                    {'label': 'Breed', 'value': 'breed'},
                    {'label': 'Age', 'value': 'age'},
                    {'label': 'Name', 'value': 'name'}
                ],
                value='suitability',
                clearable=False
            )
        ], style={'flex': '1 1 240px'})
    ], style={
        'display': 'flex',
        'gap': '16px',
        'flexWrap': 'wrap',
        'padding': '12px 0'
    }),

    html.Div(
        initial_message,
        id='dashboard-status',
        role='status',
        style={'color': initial_color, 'fontWeight': 'bold', 'padding': '10px 0'}
    ),

    html.Hr(),

    dcc.Loading(type='circle', children=[
    dash_table.DataTable(
    id='datatable-id',
    columns=[{
        "name": "Suitability Score" if i == "suitability_score" else i,
        "id": i,
        "deletable": False,
        "selectable": True
    } for i in df.columns],
    data=df.to_dict('records'),

   
    page_action='custom',
    page_current=0,
    page_size=PAGE_SIZE,
    page_count=initial_page_count,
    sort_action='custom',
    sort_mode='single',
    sort_by=[],
    filter_action='custom',
    filter_query='',

    
    row_selectable='single',
    selected_rows=[0] if not df.empty else [],

    style_table={'overflowX': 'auto'},
    style_cell={'textAlign': 'left', 'padding': '6px', 'whiteSpace': 'normal', 'height': 'auto'},
    style_header={'fontWeight': 'bold'}
)
]),

    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
        style={'display': 'flex', 'gap': '20px', 'flexWrap': 'wrap'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',
            style={'flex': '1 1 500px', 'minWidth': '300px'},

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            style={'flex': '1 1 500px', 'minWidth': '300px'},
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################

# Clear every additional filter when Reset (All) is selected
@app.callback(
    Output('breed-filter', 'value'),
    Output('location-filter', 'value'),
    Output('sort-field', 'value'),
    Output('datatable-id', 'filter_query'),
    Output('datatable-id', 'sort_by'),
    Input('filter-type', 'value')
)
def reset_all_filters(filter_type):
    if filter_type != 'reset':
        return (no_update,) * 5
    return None, [], 'suitability', '', []


# Change both age dropdowns when the unit changes or Reset is selected
@app.callback(
    Output('youngest-age', 'options'),
    Output('oldest-age', 'options'),
    Output('youngest-age', 'value'),
    Output('oldest-age', 'value'),
    Input('age-unit', 'value'),
    Input('filter-type', 'value')
)
def update_age_dropdowns(age_unit, filter_type):
    if ctx.triggered_id == 'filter-type' and filter_type != 'reset':
        return (no_update,) * 4
    options = build_age_options(age_unit)
    return options, options, 0, options[-1]['value']


# Return to the first page whenever a dashboard filter changes
@app.callback(
    Output('datatable-id', 'page_current'),
    Input('filter-type', 'value'),
    Input('breed-filter', 'value'),
    Input('age-unit', 'value'),
    Input('youngest-age', 'value'),
    Input('oldest-age', 'value'),
    Input('location-filter', 'value'),
    Input('sort-field', 'value'),
    Input('datatable-id', 'filter_query'),
    Input('datatable-id', 'sort_by')
)
def reset_table_page(*_filters):
    return 0


# Update the table using limited database reads
@app.callback(
    Output('datatable-id', 'data'),
    Output('datatable-id', 'selected_rows'),
    Output('dashboard-status', 'children'),
    Output('dashboard-status', 'style'),
    Output('datatable-id', 'page_count'),
    Input('filter-type', 'value'),
    Input('breed-filter', 'value'),
    Input('age-unit', 'value'),
    Input('youngest-age', 'value'),
    Input('oldest-age', 'value'),
    Input('location-filter', 'value'),
    Input('sort-field', 'value'),
    Input('datatable-id', 'page_current'),
    Input('datatable-id', 'page_size'),
    Input('datatable-id', 'filter_query'),
    Input('datatable-id', 'sort_by')
)
def update_dashboard(
    filter_type,
    selected_breed,
    age_unit,
    youngest_age,
    oldest_age,
    location_filter,
    sort_field,
    page_current,
    page_size,
    table_filter_query,
    sort_by
):
    if db is None:
        error_style = {'color': '#9b1c1c', 'fontWeight': 'bold'}
        return [], [], database_error, error_style, 0

    safe_page = page_current if isinstance(page_current, int) else 0
    safe_page_size = page_size if isinstance(page_size, int) and page_size > 0 else PAGE_SIZE
    selected_display_range, selected_age_range, unit = selected_age_ranges(
        age_unit,
        youngest_age,
        oldest_age
    )
    require_location = 'valid' in (location_filter or [])

    try:
        query = build_database_query(
            filter_type,
            selected_breed=selected_breed,
            age_range=selected_age_range,
            require_location=require_location,
            table_filter_query=table_filter_query
        )
        sort_spec, processing_sort, table_sort_direction = table_sort_settings(
            sort_field,
            sort_by
        )
    except ValueError:
        logger.exception('A rescue profile could not be validated.')
        error_style = {'color': '#9b1c1c', 'fontWeight': 'bold'}
        message = 'A dashboard filter could not be applied.'
        return [], [], message, error_style, 0

    try:
        db.client.admin.command('ping')
        profile = RESCUE_PROFILES.get(filter_type)
        if profile is not None and processing_sort == 'suitability':
            results = db.read(query, limit=MAX_SCORED_RESULTS)
            all_records = process_candidates(
                results,
                filter_type,
                selected_breed=selected_breed,
                age_range=selected_age_range,
                require_location=require_location,
                sort_field='suitability'
            )
            if table_sort_direction == 'asc':
                all_records.reverse()
            total_matches = len(all_records)
            start = safe_page * safe_page_size
            records = all_records[start:start + safe_page_size]
        else:
            total_matches = db.count(query)
            results = db.read(
                query,
                sort=sort_spec,
                skip=safe_page * safe_page_size,
                limit=safe_page_size
            )
            records = process_candidates(
                results,
                filter_type,
                selected_breed=selected_breed,
                age_range=selected_age_range,
                require_location=require_location,
                sort_field=processing_sort
            )
    except PyMongoError:
        logger.exception('The selected rescue filter could not be loaded.')
        error_style = {'color': '#9b1c1c', 'fontWeight': 'bold'}
        message = 'The database could not be reached. Check MongoDB and try again.'
        return [], [], message, error_style, 0

    page_count = math.ceil(total_matches / safe_page_size) if total_matches else 0
    filter_name = FILTER_NAMES.get(filter_type, FILTER_NAMES['reset'])
    if not total_matches:
        warning_style = {'color': '#8a5a00', 'fontWeight': 'bold'}
        return [], [], 'No animals matched the selected criteria.', warning_style, 0

    if filter_type == 'reset':
        message = f'Displaying {total_matches:,} available shelter animals.'
    else:
        message = f'Displaying {total_matches:,} animals matching {filter_name}.'
        if processing_sort == 'suitability' and table_sort_direction == 'desc':
            message += ' Highest suitability scores are shown first.'

    active_filters = []
    if selected_breed:
        active_filters.append(f'breed: {selected_breed}')
    if selected_age_range is not None:
        active_filters.append(
            f'age: {selected_display_range[0]:g} to '
            f'{selected_display_range[1]:g} {unit}'
        )
    if require_location:
        active_filters.append('valid location required')
    if active_filters:
        message += ' Additional filters: ' + ', '.join(active_filters) + '.'

    logger.info(
        "Filter '%s' returned page %s of %s records.",
        filter_type,
        safe_page + 1,
        total_matches
    )
    success_style = {'color': '#185f37', 'fontWeight': 'bold'}
    selected_rows = [0] if records else []
    return records, selected_rows, message, success_style, page_count

@app.callback(
    Output('graph-id', 'children'),
    Input('filter-type', 'value'),
    Input('breed-filter', 'value'),
    Input('age-unit', 'value'),
    Input('youngest-age', 'value'),
    Input('oldest-age', 'value'),
    Input('location-filter', 'value'),
    Input('datatable-id', 'filter_query')
)
def update_graphs(
    filter_type,
    selected_breed,
    age_unit,
    youngest_age,
    oldest_age,
    location_filter,
    table_filter_query
):
    if db is None:
        return html.Div(database_error)

    _, selected_age_range, _ = selected_age_ranges(
        age_unit,
        youngest_age,
        oldest_age
    )
    require_location = 'valid' in (location_filter or [])
    try:
        query = build_database_query(
            filter_type,
            selected_breed=selected_breed,
            age_range=selected_age_range,
            require_location=require_location,
            table_filter_query=table_filter_query
        )
        breed_counts = db.breed_counts(query)
    except (ValueError, PyMongoError):
        logger.exception('Breed totals could not be loaded.')
        return html.Div('Breed data could not be loaded.')

    if not breed_counts:
        return html.Div('No breed data is available for the chart.')

    top_n = 6
    top_breeds = breed_counts[:top_n]
    other_count = sum(item['count'] for item in breed_counts[top_n:])
    final_counts = list(top_breeds)
    if other_count > 0:
        final_counts.append({'breed': 'Other', 'count': other_count})
    final_df = pd.DataFrame(final_counts)

    fig = px.pie(
        final_df,
        names='breed',
        values='count',
        title='Animals by Breed'
)

    return dcc.Graph(figure=fig, config={'displayModeBar': False})



# Update the map for the selected table row
@app.callback(
    Output('map-id', 'children'),
    Input('datatable-id', 'derived_virtual_data'),
    Input('datatable-id', 'derived_virtual_selected_rows')
)
def update_map(view_data, selected_rows):
    records = df.to_dict('records') if view_data is None else view_data

    if not records:
        return html.Div('No animal records are available for the map.')
    if not selected_rows:
        return html.Div('Select an animal from the table to view its location.')

    row_number = selected_rows[0]
    if row_number < 0 or row_number >= len(records):
        return html.Div('The selected animal is no longer visible in the table.')

    animal = records[row_number]
    try:
        latitude = float(animal.get('location_lat'))
        longitude = float(animal.get('location_long'))
    except (TypeError, ValueError):
        return html.Div('Location information is unavailable for this animal.')

    if not coordinates_are_valid(latitude, longitude):
        return html.Div('Location information is invalid for this animal.')

    breed = animal.get('breed') or 'Breed unavailable'
    name = animal.get('name') or 'Unnamed animal'

    return dl.Map(
        style={'width': '100%', 'height': '500px'},
        center=[latitude, longitude],
        zoom=10,
        children=[
            dl.TileLayer(id='base-layer-id'),
            dl.Marker(
                position=[latitude, longitude],
                children=[
                    dl.Tooltip(breed),
                    dl.Popup([html.H4('Animal Name'), html.P(name)])
                ]
            )
        ]
    )


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run(jupyter_mode='external', debug=False)